# DesignBridge — Flux Dev LoRA Finetune
**目標**：讓 FLUX.1-dev 學習台灣室內設計美學（domain adaptation），不使用 trigger word。

**框架**：[AI-Toolkit (ostris)](https://github.com/ostris/ai-toolkit)

**需求**：
- Colab A100（推薦）或 T4（慢但可跑）
- HuggingFace token（需申請 FLUX.1-dev 存取權）
- Google Drive 上傳好的資料集 ZIP

In [ ]:
# ── Step 0: 確認 GPU ──────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Step 1: 安裝 AI-Toolkit ───────────────────────────────────────────────────
import os

if not os.path.exists('/content/ai-toolkit'):
    !git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
    !git -C /content/ai-toolkit submodule update --init --recursive

%cd /content/ai-toolkit
!pip install -q -r requirements.txt
!pip install -q bitsandbytes>=0.43.0 scipy

In [ ]:
# ── Step 2: HuggingFace 登入（需要 FLUX.1-dev 存取權）───────────────────────
# 前往 https://huggingface.co/settings/tokens 產生 token
# 前往 https://huggingface.co/black-forest-labs/FLUX.1-dev 申請存取
from huggingface_hub import login
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # <-- 換成你的 token
login(token=HF_TOKEN)

In [ ]:
# ── Step 3: 掛載 Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# 確認 Drive 結構
!ls /content/drive/MyDrive/

In [ ]:
# ── Step 4: 解壓資料集 ────────────────────────────────────────────────────────
# 請先把 designbridge_dataset.zip 上傳到 Google Drive 根目錄
DATASET_ZIP = "/content/drive/MyDrive/designbridge_dataset.zip"  # <-- 調整路徑
DATASET_DIR = "/content/dataset"

if not os.path.exists(DATASET_DIR):
    !unzip -q "{DATASET_ZIP}" -d "{DATASET_DIR}"

# 驗證資料集
import glob
images = glob.glob(f"{DATASET_DIR}/**/*.jpg", recursive=True) + \
         glob.glob(f"{DATASET_DIR}/**/*.png", recursive=True)
captions = glob.glob(f"{DATASET_DIR}/**/*.txt", recursive=True)

print(f"圖片數量: {len(images)}")
print(f"Caption 數量: {len(captions)}")

# 顯示缺少 caption 的圖片
missing = []
for img in images[:5]:
    txt = os.path.splitext(img)[0] + '.txt'
    if not os.path.exists(txt):
        missing.append(img)
if missing:
    print(f"\n⚠️  缺少 caption: {len(missing)} 張（顯示前 5 筆）")
    for f in missing[:5]: print(" ", f)
else:
    print("\n✅ 所有圖片都有對應 caption")

In [ ]:
# ── Step 5: 產生訓練 Config ───────────────────────────────────────────────────
import yaml, os

# ====== 可調參數 ======
RUN_NAME      = "designbridge_v1"
LORA_RANK     = 16        # 16 or 32（32 效果更強但更慢）
TRAIN_STEPS   = 1000      # POC 用 500-1000；正式用 2000-3000
LEARNING_RATE = 1e-4
BATCH_SIZE    = 1
GRAD_ACCUM    = 4         # 等效 batch_size=4
SAMPLE_EVERY  = 250       # 每 N steps 出一組預覽圖
SAVE_EVERY    = 500       # 每 N steps 存一個 checkpoint
# =====================

OUTPUT_DIR = f"/content/ai-toolkit/output/{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

config = {
    "job": "extension",
    "config": {
        "name": RUN_NAME,
        "process": [{
            "type": "sd_trainer",
            "training_folder": OUTPUT_DIR,
            "device": "cuda:0",
            "network": {
                "type": "lora",
                "linear": LORA_RANK,
                "linear_alpha": LORA_RANK
            },
            "save": {
                "dtype": "float16",
                "save_every": SAVE_EVERY,
                "max_step_saves_to_keep": 3
            },
            "datasets": [{
                "folder_path": DATASET_DIR,
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": [512, 768, 1024]
            }],
            "train": {
                "batch_size": BATCH_SIZE,
                "steps": TRAIN_STEPS,
                "gradient_accumulation_steps": GRAD_ACCUM,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": LEARNING_RATE,
                "ema_config": {
                    "use_ema": True,
                    "ema_decay": 0.99
                },
                "dtype": "bf16"
            },
            "model": {
                "name_or_path": "black-forest-labs/FLUX.1-dev",
                "is_flux": True,
                "quantize": True   # NF4 量化，省 VRAM（T4 必開，A100 可關）
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": SAMPLE_EVERY,
                "width": 1024,
                "height": 1024,
                "prompts": [
                    "a modern Taiwanese living room, clean geometric lines, natural wood flooring, white sofa, floor-to-ceiling windows, neutral gray and white tones, bright airy space",
                    "a Japanese zen bedroom, wood lattice shoji screen, tatami platform, muted earthy tones, serene minimal atmosphere, soft natural light",
                    "a Nordic style open kitchen, white cabinets, light wood countertop, pendant lights, cozy minimalist Scandinavian design",
                    "a luxury Taiwanese master bedroom, marble accent wall, gold brass fixtures, velvet headboard, refined elegant atmosphere"
                ],
                "neg": "",
                "seed": 42,
                "walk_seed": True,
                "guidance_scale": 4,
                "sample_steps": 20
            }
        }]
    },
    "meta": {
        "name": "DesignBridge Flux LoRA",
        "version": "1.0"
    }
}

CONFIG_PATH = f"/content/ai-toolkit/config/{RUN_NAME}.yaml"
os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"✅ Config 已儲存：{CONFIG_PATH}")
print(f"   Steps: {TRAIN_STEPS}")
print(f"   LoRA Rank: {LORA_RANK}")
print(f"   有效 Batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Output: {OUTPUT_DIR}")

In [ ]:
# ── Step 6: 開始訓練 ──────────────────────────────────────────────────────────
# 預計時間（A100）：
#   500  steps ~ 15 min
#   1000 steps ~ 30 min
#   2000 steps ~ 60 min

%cd /content/ai-toolkit
!python run.py "{CONFIG_PATH}"

In [ ]:
# ── Step 7: 查看訓練結果與預覽圖 ─────────────────────────────────────────────
import glob
from IPython.display import Image, display

# 列出所有 checkpoint
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/*.safetensors"))
print("Checkpoints:")
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f"  {os.path.basename(ckpt)} ({size_mb:.0f} MB)")

# 顯示最新一批預覽圖
sample_dirs = sorted(glob.glob(f"{OUTPUT_DIR}/samples/*"))
if sample_dirs:
    latest = sample_dirs[-1]
    print(f"\n預覽圖（{os.path.basename(latest)}）:")
    for img_path in sorted(glob.glob(f"{latest}/*.png")):
        display(Image(img_path, width=512))

In [ ]:
# ── Step 8: 存到 Google Drive ─────────────────────────────────────────────────
DRIVE_OUTPUT = f"/content/drive/MyDrive/flux_lora/{RUN_NAME}"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# 只複製最終 LoRA（最後一個 safetensors）
if checkpoints:
    final_lora = checkpoints[-1]
    dest = os.path.join(DRIVE_OUTPUT, os.path.basename(final_lora))
    !cp "{final_lora}" "{dest}"
    print(f"✅ LoRA 已存到 Drive: {dest}")
    print(f"   大小: {os.path.getsize(dest) / 1e6:.0f} MB")

# 複製所有預覽圖
if sample_dirs:
    samples_dest = os.path.join(DRIVE_OUTPUT, "samples")
    !cp -r "{OUTPUT_DIR}/samples" "{samples_dest}"
    print(f"✅ 預覽圖已存到 Drive: {samples_dest}")

## 使用 LoRA

訓練完成的 `.safetensors` 可以直接用在：

**ComfyUI**
```
ComfyUI/models/loras/designbridge_v1.safetensors
```
在 workflow 加入 `Load LoRA` 節點，strength 建議 **0.8~1.0**。

**AUTOMATIC1111 / Forge**
```
models/Lora/designbridge_v1.safetensors
```
Prompt 裡加 `<lora:designbridge_v1:0.9>`

**建議 Prompt 格式**
```
a [style] Taiwanese [room type], [key elements], [materials], [lighting], [mood], high quality interior render
```
範例：
```
a modern Taiwanese living room, open plan layout, natural wood flooring, 
floor-to-ceiling windows, minimalist furniture, neutral warm tones, 
soft afternoon light, high quality interior render
```